## Unstructured Data - Delta Lake

In this notebook, we use Delta tables to store metadata extracted from unstructured data sources. Structured and semi-structured datasets are excluded from this workflow, as they are no longer stored in the Trusted Zone in MinIO.

**Importing Useful Libraries**

In [8]:
import os
import boto3
import duckdb
import hashlib
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl
from PIL import Image
from PIL.ExifTags import TAGS
import pandas as pd
import json
import io

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [9]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [10]:
def extract_timestamp_from_filename(filename):
    # Strip extension and split by underscore
    name_part = os.path.splitext(filename)[0]
    raw_ts = name_part.split('_')[-1]

    try:
        # Convert string epoch to a readable datetime object
        dt_object = datetime.fromtimestamp(int(raw_ts))
        return dt_object
    except (ValueError, IndexError):
        # Fallback if the filename doesn't follow the pattern
        return datetime.now()
        
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

**Unstructured Data**

In [11]:
# ----------------------------
# CONFIG
# ----------------------------
CATALOG_PATH = "s3://trusted-zone/persistent-landing/structured/file_catalog/"
BUCKET = "trusted-zone"
PREFIX = "persistent-landing/structured/file_catalog/"

EXPECTED_COLUMNS = [
    "file_id",
    "landing_file_id",
    "source_type",
    "file_type",
    "event_time",
    "record_count",
    "metadata_blob",
    "processed_at"
]

# ----------------------------
# ENSURE DELTA CATALOG EXISTS
# ----------------------------
def ensure_catalog_exists(sample_df):
    try:
        DeltaTable(CATALOG_PATH, storage_options=storage_options)
        print("Delta table already exists.")
        return False
    except Exception:
        print("Creating Delta table...")

        write_deltalake(
            CATALOG_PATH,
            sample_df,
            mode="overwrite",
            schema_mode="merge",
            storage_options=storage_options
        )
        return True


# ----------------------------
# CREATE SAMPLE SCHEMA
# ----------------------------
sample_df = pd.DataFrame([{
    "file_id": "",
    "landing_file_id": "",
    "source_type": "",
    "file_type": "Image",
    "event_time": str(pd.Timestamp.now()),
    "record_count": 1,
    "metadata_blob": "{}",
    "processed_at": str(pd.Timestamp.now())
}])

ensure_catalog_exists(sample_df)


# ----------------------------
# IMAGE PROCESSING
# ----------------------------
def process_image(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        page_records = []

        for obj in page.get("Contents", []):
            src_key = obj["Key"]

            # skip folders / empty files
            if src_key.endswith("/") or obj["Size"] == 0:
                continue

            metadata_blob = {}

            try:
                response = s3.get_object(Bucket=bucket, Key=src_key)
                content = response["Body"].read()

                img = Image.open(io.BytesIO(content))
                width, height = img.size
                metadata = response.get("Metadata", {})

                metadata_blob = {
                    "label": metadata.get("label"),
                    "url": metadata.get("url"),
                    "file_size_bytes": obj["Size"],
                    "content_type": response.get("ContentType"),
                    "width": width,
                    "height": height,
                    "aspect_ratio": round(width / height, 2) if height > 0 else 0,
                    "image_mode": img.mode,
                    "is_corrupted": False,
                    "md5": hashlib.md5(content).hexdigest()
                }

            except Exception as e:
                print(f"Error parsing image {src_key}: {e}")

                metadata_blob = {
                    "label": None,
                    "url": None,
                    "file_size_bytes": obj["Size"],
                    "content_type": None,
                    "width": 0,
                    "height": 0,
                    "aspect_ratio": 0,
                    "image_mode": "unknown",
                    "is_corrupted": True,
                    "error_msg": str(e),
                    "md5": None
                }

            filename = os.path.basename(src_key)
            name, _ = os.path.splitext(filename)

            metadata_row = {
                "file_id": filename,
                "landing_file_id": f"landing-zone/persistent-landing/unstructured/image/{name}.jpg",
                "source_type": metadata.get("source", "unknown"),
                "file_type": "Image",
                "event_time": str(extract_timestamp_from_filename(filename)),
                "record_count": 1,
                "metadata_blob": json.dumps(metadata_blob, ensure_ascii=False),
                "processed_at": str(pd.Timestamp.now())
            }

            page_records.append(metadata_row)

        # ----------------------------
        # WRITE BATCH TO DELTA
        # ----------------------------
        if page_records:

            df = pd.DataFrame(page_records)

            # enforce schema consistency
            for col in EXPECTED_COLUMNS:
                if col not in df.columns:
                    df[col] = None

            df = df[EXPECTED_COLUMNS]

            # fix null-safe defaults
            df = df.fillna({
                "file_id": "",
                "landing_file_id": "",
                "source_type": "unknown",
                "file_type": "Image",
                "metadata_blob": "{}"
            })

            write_deltalake(
                CATALOG_PATH,
                df,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )

            print(f"Batch uploaded: {len(df)} images")

Delta table already exists.


In [12]:
process_image("trusted-zone","unstructured/image")

Batch uploaded: 1000 images
Batch uploaded: 93 images


In [13]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [14]:
query = "SELECT * FROM delta_scan('s3://trusted-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,landing_file_id,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,image_1779622088400.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.624756,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.624792
1,image_1779622088466.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.631854,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.631880
2,image_1779622088542.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.640299,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.640332
3,image_1779622088594.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.649456,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.649488
4,image_1779622088656.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.657936,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.657966
...,...,...,...,...,...,...,...,...
1088,image_1779622088042.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.211659,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.211699
1089,image_1779622088105.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.217492,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.217515
1090,image_1779622088164.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.226424,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.226447
1091,image_1779622088234.png,landing-zone/persistent-landing/unstructured/i...,unknown,Image,2026-05-24 12:00:41.236939,1,"{""label"": null, ""url"": null, ""file_size_bytes""...",2026-05-24 12:00:41.236976
